In [ ]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
dbutils = WorkspaceClient().dbutils()

In [ ]:
kelp_project_file = dbutils.widgets.get("kelp_project_file")
kelp_target = dbutils.widgets.get("kelp_target")
full_refresh = dbutils.widgets.get("full_refresh").lower() == "true"

In [ ]:
from pyspark.sql import SparkSession

import kelp.tables as kt

spark = SparkSession.active()

_ = kt.init(kelp_project_file, target=kelp_target)

In [ ]:
from datetime import date, datetime, timedelta
from typing import Any

from pyspark.sql import DataFrame
from pyspark.sql import functions as f


@kt.materialized(name="bronze_customers_mat")
def bronze_customers_mat(ctx: kt.MaterializedContext) -> DataFrame:
    bronze_customers_seed = [
        {
            "user_id": "U100",
            "first_name": "Ana",
            "last_name": "Nguyen",
            "country": "US",
            "email": "ana.nguyen@example.com",
            "customer_updated_at": datetime(2026, 5, 31, 10, 0, 0),
            "ingest_date": date(2026, 5, 31),
        },
        {
            "user_id": "U200",
            "first_name": "Bruno",
            "last_name": "Silva",
            "country": "BR",
            "email": "bruno.silva@example.com",
            "customer_updated_at": datetime(2026, 5, 31, 10, 15, 0),
            "ingest_date": date(2026, 5, 31),
        },
        {
            "user_id": "U300",
            "first_name": "Chloe",
            "last_name": "Martin",
            "country": "FR",
            "email": "chloe.martin@example.com",
            "customer_updated_at": datetime(2026, 5, 31, 10, 30, 0),
            "ingest_date": date(2026, 5, 31),
        },
    ]

    return spark.createDataFrame(bronze_customers_seed)


@kt.materialized(name="bronze_orders_mat")
def bronze_orders_mat(ctx: kt.MaterializedContext) -> DataFrame:
    first_load = [
        {
            "order_nk": "O-1001",
            "user_id": "U100",
            "product": "Laptop",
            "quantity": 1,
            "unit_price": 1200.0,
            "order_date": date(2026, 5, 31),
            "order_updated_at": datetime(2026, 5, 31, 11, 0, 0),
            "ingest_date": date(2026, 5, 31),
        },
        {
            "order_nk": "O-1002",
            "user_id": "U200",
            "product": "Keyboard",
            "quantity": 2,
            "unit_price": 90.0,
            "order_date": date(2026, 5, 31),
            "order_updated_at": datetime(2026, 5, 31, 11, 10, 0),
            "ingest_date": date(2026, 5, 31),
        },
        {
            "order_nk": "O-1003",
            "user_id": "U300",
            "product": "Mouse",
            "quantity": 1,
            "unit_price": 45.0,
            "order_date": date(2026, 6, 1),
            "order_updated_at": datetime(2026, 6, 1, 9, 0, 0),
            "ingest_date": date(2026, 6, 1),
        },
    ]

    if not ctx.is_incremental():
        return spark.createDataFrame(first_load)

    # Unlimited re-runs: generate fresh natural keys and timestamps each run.
    now_ts = datetime.now()
    run_token = now_ts.strftime("%Y%m%d%H%M%S%f")
    incremental_load = [
        {
            "order_nk": f"O-{run_token}-001",
            "user_id": "U100",
            "product": "Monitor",
            "quantity": 1,
            "unit_price": 320.0,
            "order_date": now_ts.date(),
            "order_updated_at": now_ts,
            "ingest_date": now_ts.date(),
        },
        {
            "order_nk": f"O-{run_token}-002",
            "user_id": "U200",
            "product": "Dock",
            "quantity": 1,
            "unit_price": 210.0,
            "order_date": now_ts.date(),
            "order_updated_at": now_ts + timedelta(minutes=1),
            "ingest_date": now_ts.date(),
        },
    ]
    return spark.createDataFrame(incremental_load)


@kt.materialized(
    name="silver_customers_refined_mat",
    depends_on=["bronze_customers_mat"],
)
def silver_customers_refined(ctx: kt.MaterializedContext) -> DataFrame:
    source_df = spark.read.table(kt.ref("bronze_customers_mat"))

    if ctx.is_incremental():
        max_customer_ts = (
            ctx.spark.table(ctx.this)
            .agg(f.max("customer_updated_at").alias("max_customer_updated_at"))
            .collect()[0]["max_customer_updated_at"]
        )
        if max_customer_ts is not None:
            source_df = source_df.filter(f.col("customer_updated_at") > f.lit(max_customer_ts))

    return (
        source_df.withColumn(
            "full_name",
            f.concat_ws(" ", f.trim(f.col("first_name")), f.trim(f.col("last_name"))),
        )
        .withColumn("country", f.upper(f.trim(f.col("country"))))
        .withColumn("email", f.lower(f.trim(f.col("email"))))
        .select(
            "user_id",
            "full_name",
            "country",
            "email",
            "customer_updated_at",
            "ingest_date",
        )
    )


@kt.materialized(
    name="silver_orders_refined_mat",
    depends_on=["bronze_orders_mat"],
)
def silver_orders_refined(ctx: kt.MaterializedContext) -> DataFrame:
    source_df = spark.read.table(kt.ref("bronze_orders_mat"))

    if ctx.is_incremental():
        max_order_ts = (
            ctx.spark.table(ctx.this)
            .agg(f.max("order_updated_at").alias("max_order_updated_at"))
            .collect()[0]["max_order_updated_at"]
        )
        if max_order_ts is not None:
            source_df = source_df.filter(f.col("order_updated_at") > f.lit(max_order_ts))

    return (
        source_df.withColumn("line_amount", f.col("quantity") * f.col("unit_price"))
        .filter(f.col("quantity") > 0)
        .filter(f.col("unit_price") >= 0)
        .select(
            "order_nk",
            "user_id",
            "product",
            "quantity",
            "unit_price",
            "line_amount",
            "order_date",
            "order_updated_at",
            "ingest_date",
        )
    )


@kt.materialized(
    name="gold_daily_sales_mat",
    depends_on=["silver_customers_refined_mat", "silver_orders_refined_mat"],
)
def gold_daily_sales(ctx: kt.MaterializedContext) -> DataFrame:
    orders_df = spark.read.table(kt.ref("silver_orders_refined_mat"))
    customers_df = spark.read.table(kt.ref("silver_customers_refined_mat"))

    if ctx.is_incremental():
        max_order_date = (
            ctx.spark.table(ctx.this)
            .agg(f.max("order_date").alias("max_order_date"))
            .collect()[0]["max_order_date"]
        )
        if max_order_date is not None:
            orders_df = orders_df.filter(f.col("order_date") >= f.lit(max_order_date))

    return (
        orders_df.alias("o")
        .join(customers_df.alias("c"), on="user_id", how="left")
        .groupBy(f.col("o.order_date").alias("order_date"), f.col("c.country").alias("country"))
        .agg(
            f.countDistinct("o.order_nk").alias("order_count"),
            f.round(f.sum("o.line_amount"), 2).alias("gross_revenue"),
            f.sum("o.quantity").alias("total_units"),
            f.max("o.order_updated_at").alias("max_order_updated_at"),
        )
    )

In [ ]:
from kelp.tables.materialization.runner import Runner

runner = Runner()
planned_models = runner.plan_all()
print("📋 Runner execution plan:")
for idx, model_name in enumerate(planned_models, start=1):
    print(f"  {idx}. {model_name}")

In [ ]:
runner.run_all(full_refresh=full_refresh)

runlog_rows = [
    {
        "model": e.model,
        "status": e.status,
        "started_at": str(e.started_at),
        "finished_at": str(e.finished_at),
        "duration_seconds": round(e.duration_seconds, 3),
        "error": e.error,
    }
    for e in runner.runlog.entries
]

schema = "model STRING, status STRING, started_at STRING, finished_at STRING, duration_seconds DOUBLE, error STRING"

print("\n🧾 Runner runlog")
spark.createDataFrame(runlog_rows,schema=schema).orderBy("model").show(truncate=False)